In [2]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [5]:
spark=SparkSession.builder.appName("IRIS_MultiClass").getOrCreate()

In [6]:
iris_df=spark.read.csv("iris.csv", header=True, inferSchema=True)

In [8]:
# Handle categorical features (if applicable)
indexer=StringIndexer(inputCol="species", outputCol="label").fit(iris_df)
iris_df=indexer.transform(iris_df)

In [9]:
#Assemble features
assembler=VectorAssembler(inputCols=["sepal_length","sepal_width","petal_length","petal_width"],outputCol="features")
iris_df=assembler.transform(iris_df)

In [15]:
#Split the data
train_df,test_df=iris_df.randomSplit([0.8,0.2],seed=42)

In [16]:
#Create and train the model
rf=RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10)
model=rf.fit(train_df)

In [18]:
#Evaluate the model
predictions=model.transform(test_df)
evaluator=MulticlassClassificationEvaluator(metricName="accuracy", labelCol="label", predictionCol="prediction")
accuracy=evaluator.evaluate(predictions)
print("Accuracy:", accuracy)

Accuracy: 0.9583333333333334
